In [0]:
WITH checks AS (

    -- Completeness: Region Code
    SELECT
        'region_code' AS column_name,
        'Completeness' AS quality_dimension,
        'Not Null Check' AS check_name,
        COUNT(*) AS total_rows,
        COUNT_IF(
            region_code IS NULL
            OR TRIM(region_code) = ''
        ) AS failed_rows,
        CAST(0 AS DECIMAL(8,2)) AS threshold,
        'Region code must not be null or blank' AS notes
    FROM pondolens.dev_silver.unemployment_rate

    UNION ALL

    -- Completeness: Region Name
    SELECT
        'region_name',
        'Completeness',
        'Not Null Check',
        COUNT(*),
        COUNT_IF(
            region_name IS NULL
            OR TRIM(region_name) = ''
        ),
        CAST(0 AS DECIMAL(8,2)),
        'Region name must not be null or blank'
    FROM pondolens.dev_silver.unemployment_rate

    UNION ALL

    -- Validity: Year
    SELECT
        'year',
        'Validity',
        'Year Range Check',
        COUNT(*),
        COUNT_IF(
            year IS NULL
            OR year NOT BETWEEN 2021 AND 2025
        ),
        CAST(0 AS DECIMAL(8,2)),
        'Year must be between 2021 and 2025'
    FROM pondolens.dev_silver.unemployment_rate

    UNION ALL

    -- Validity: Unemployment Rate
    SELECT
        'unemployment_rate',
        'Validity',
        'Rate Range Check',
        COUNT(*),
        COUNT_IF(
            unemployment_rate < 0
            OR unemployment_rate > 100
        ),
        CAST(0 AS DECIMAL(8,2)),
        'Unemployment rate must be between 0 and 100'
    FROM pondolens.dev_silver.unemployment_rate

    UNION ALL

    -- Validity: Surrogate ID
    SELECT
        'unemployment_rate_id',
        'Validity',
        'ID Range Check',
        COUNT(*),
        COUNT_IF(
            unemployment_rate_id IS NULL
            OR unemployment_rate_id < 3000000
        ),
        CAST(0 AS DECIMAL(8,2)),
        'Unemployment rate IDs must start at 3000000'
    FROM pondolens.dev_silver.unemployment_rate

    UNION ALL

    -- Uniqueness: Surrogate ID
    SELECT
        'unemployment_rate_id',
        'Uniqueness',
        'Duplicate ID Check',
        COUNT(*),
        COUNT(*) - COUNT(DISTINCT unemployment_rate_id),
        CAST(0 AS DECIMAL(8,2)),
        'Every unemployment_rate_id must be unique'
    FROM pondolens.dev_silver.unemployment_rate

    UNION ALL

    -- Uniqueness: Business Key
    SELECT
        'year, region_code',
        'Uniqueness',
        'Duplicate Business Key Check',
        COUNT(*),
        COUNT(*) - COUNT(DISTINCT
            STRUCT(year, region_code)
        ),
        CAST(0 AS DECIMAL(8,2)),
        'Each year and region_code combination must be unique'
    FROM pondolens.dev_silver.unemployment_rate
),

results AS (
    SELECT
        'pondolens.dev_silver.unemployment_rate' AS table_name,
        column_name,
        quality_dimension,
        check_name,
        total_rows,
        failed_rows,

        CAST(
            CASE
                WHEN total_rows = 0 THEN 0
                ELSE failed_rows * 100.0 / total_rows
            END AS DECIMAL(8,2)
        ) AS failure_percentage,

        threshold,
        notes
    FROM checks
)

SELECT
    table_name,
    column_name,
    quality_dimension,
    check_name,
    total_rows,
    failed_rows,
    failure_percentage,

    CASE
        WHEN total_rows = 0 THEN 'WARN'
        WHEN failure_percentage > threshold THEN 'FAIL'
        ELSE 'PASS'
    END AS status,

    threshold,
    CURRENT_TIMESTAMP() AS check_timestamp,
    notes

FROM results;